# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Building the Global Development Monitor

### Scenario

You've just joined the analytics team at the **Global Development Observatory**, an NGO
that briefs policymakers on world development trends. Your director has been burned
before by bad visualizations, and gives you a blunt brief:

1. *"The last analyst showed me a map that made me think Greenland mattered more than
   India. Don't let that happen again."*
2. *"We're about to publish a model predicting life expectancy. If a journalist asks
   'why did the model predict THAT for Norway specifically,' I need an answer, not just
   a feature-importance bar chart for the whole model."*
3. *"Every number in our reports is an estimate. I want our numbers to LOOK like
   estimates, not like certainties."*
4. *"I want one screen I can glance at every morning that tells me what's changing."*

You'll work through the same steps a real analyst would: fix a misleading map, explain
individual model predictions, add honest uncertainty to an estimate and a trend, then
build a small coordinated-views dashboard.

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection**
are for you to answer in your own words.

## Part 0 — Setup (given)

Run this cell as-is. It loads the real dataset and real country centroids you'll be
working with.

In [1]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall numpy pandas

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/c5/31/7fc6239c12bce7e931463251cca4426c465e1876ba3cc785402ef4dd8f4e/numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/d3/dc/d2df02854aec5d47659acfb2be352eecc691845b2f86e99c84f1010a8671/pandas-3.0.6-cp311-cp311-win_amd64.whl.metadata
  Using cached pandas-3.0.6-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Obtaining dependency information for python-dateutil>=2.8.2 from https://files.pythonhosted.org/packages/ec/57/56b9bcc3c9c6a792fcbaf139543cee77261f3651ca9da0c93f5c1221264b/python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Obtaining dependency information for tzdata from https

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.75 requires PyYAML==6.0.1, but you have pyyaml 6.0 which is incompatible.


## Task 1 — Fix the Choropleth Trap (Complaint #1)

**Real-world usage:** The director was misled by a raw-count choropleth that made a
huge, sparsely-populated region look more important than it should.

**Why this technique:** Distinguishing **counts** from **rates** is the single most
important geospatial fix from the lecture — a "count" choropleth is dominated by
population size, not by the thing you actually care about.

**TODO:**
1. Build a choropleth of `total_gdp` (a count-like aggregate) using `px.choropleth`.
2. Build a second choropleth of `gdpPercap` (the rate).
3. Print the top-5 countries by each, and compare.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
import shap

import ipywidgets as widgets
from ipywidgets import interact, Dropdown

np.random.seed(0)

gapminder = px.data.gapminder()
centroids = pd.read_csv("D:\\VDUI_HANDSON\\unit2\\PPT 5 ACTIVITY\\unit2_ppt5_country_centroids_computed.csv")
gap_latest = gapminder[gapminder.year == 2007].merge(centroids, on="iso_alpha", how="left")
gap_latest = gap_latest.assign(total_gdp=gap_latest["pop"] * gap_latest.gdpPercap)

print(f"Loaded {gapminder.country.nunique()} countries, years {sorted(gapminder.year.unique())}.")
gap_latest[["country", "continent", "year", "lifeExp", "pop", "gdpPercap"]].head()

In [ ]:
# TODO 1: choropleth of total_gdp
fig_total = px.choropleth(
    gap_latest,
    locations="iso_alpha",
    color="total_gdp",
    hover_name="country",
    color_continuous_scale="Viridis",
    title="Total GDP by country (2007) — a count-like aggregate"
)
fig_total.show()

# TODO 2: choropleth of gdpPercap
fig_rate = px.choropleth(
    gap_latest,
    locations="iso_alpha",
    color="gdpPercap",
    hover_name="country",
    color_continuous_scale="Viridis",
    title="GDP per capita by country (2007) — the rate"
)
fig_rate.show()

# TODO 3: top 5 countries by each -- print both lists
top_total = gap_latest.nlargest(5, "total_gdp")[["country", "total_gdp"]]
top_rate = gap_latest.nlargest(5, "gdpPercap")[["country", "gdpPercap"]]
print("Top 5 by total GDP:\n", top_total)
print("\nTop 5 by GDP per capita:\n", top_rate)


**Reflection:** Which countries appear in one top-5 list but not the other? If you had
to pick ONE map to show the director, which would you pick, and why?

_Your answer:_

The total-GDP top 5 is dominated by huge, populous economies like the United States, China, Japan,
Germany and India — countries that rank high mainly because they have a LOT of people, not because
any individual person there is well-off. The GDP-per-capita top 5 looks completely different — small,
wealthy countries like Norway, Kuwait, Singapore, the US and Ireland show up instead, because that
list measures how well-off the average person actually is, regardless of population size. India shows
up in the total-GDP list purely because of its huge population, even though its GDP per capita is
comparatively low, which is exactly the kind of "big country looks important" trap the director
mentioned. If I had to pick one map for the director, I'd pick the GDP-per-capita (rate) map, because
it answers the question policymakers actually care about — how well-off are the people in this
country — instead of just re-drawing the population map in a different color.


## Task 2 — Explain One Specific Prediction (Complaint #2)

**Real-world usage:** The director needs to answer "why did the model predict THAT for
Norway specifically" — a global importance bar chart cannot answer a question about one
country.

**Why this technique:** SHAP local importance decomposes ONE prediction into
per-feature, signed contributions — exactly what a journalist question needs.

**TODO:**
1. Fit a `RandomForestRegressor` predicting `lifeExp` from `["gdpPercap", "pop", "year"]`
   on the full `gapminder` dataset.
2. Build a `shap.TreeExplainer` and compute `shap_values` for all rows.
3. Find the row index for Norway, 2007, and plot its 3 SHAP values as a diverging
   horizontal bar chart (blue if positive, red if negative).

In [ ]:
X_model = gapminder[["gdpPercap", "pop", "year"]].values
y_model = gapminder["lifeExp"].values

# TODO 1: fit the model
rf = RandomForestRegressor(n_estimators=200, random_state=0)
rf.fit(X_model, y_model)

# TODO 2: SHAP explainer + values
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_model)

# TODO 3: find Norway's row, plot its SHAP contributions
row_idx = gapminder[(gapminder.country == "Norway") & (gapminder.year == 2007)].index[0]
vals = shap_values[row_idx]
feature_names = ["gdpPercap", "pop", "year"]

fig, ax = plt.subplots(figsize=(7, 3.5))
# plot vals as a diverging horizontal bar chart here
colors = ["#3b6ea5" if v >= 0 else "#b5432e" for v in vals]
ax.barh(feature_names, vals, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("SHAP value (impact on predicted lifeExp)")
ax.set_title("Local SHAP contributions — Norway, 2007")

plt.show()


**Reflection:** Which feature contributed most POSITIVELY to Norway's predicted life
expectancy, and which (if any) contributed negatively? Is this the same answer you'd get
from the model's GLOBAL feature importance alone?

_Your answer:_

For Norway in 2007, `gdpPercap` is by far the biggest positive contributor — Norway's GDP per capita
is very high, and the model pushes its life expectancy prediction up because of that. `year` also
contributes a small positive push (2007 is late in the dataset, and life expectancy trends upward over
time globally), while `pop` contributes only a tiny amount either way since Norway's population isn't
unusual. This is NOT necessarily what you'd get from global feature importance alone — a global
importance chart just tells you `gdpPercap` matters a lot across ALL rows on average, but it can't
tell you the direction or size of its effect for one specific country. Norway could easily be a case
where `gdpPercap`'s effect is much larger than the global average effect, and SHAP is what actually
lets us say that for this one country specifically, instead of guessing from the aggregate number.


## Task 3 — Make an Estimate Look Like an Estimate (Complaint #3)

**Real-world usage:** Every number in the report should visually communicate its own
uncertainty, not look like a fixed fact.

**Why this technique:** Bootstrap confidence intervals + error bars turn a bare point
estimate into an honest range.

**TODO:**
1. Write `bootstrap_ci(values, n_boot=2000)` that resamples `values` with replacement
   `n_boot` times, computes the mean each time, and returns `(mean, ci_low, ci_high)`
   using the 2.5th and 97.5th percentiles of the bootstrap means.
2. Compute a 95% CI for mean `lifeExp` in 2007 for **two** continents of your choice.
3. Plot both as error bars.

In [ ]:
def bootstrap_ci(values, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    # TODO 1: resample `values` with replacement n_boot times, take the mean each time,
    # then return (overall mean, 2.5th percentile of boot means, 97.5th percentile)
    boot_means = np.empty(n_boot)
    n = len(values)
    for i in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boot_means[i] = sample.mean()
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    return values.mean(), ci_low, ci_high

# TODO 2: compute CIs for two continents of your choice
continent_a, continent_b = "Africa", "Europe"  # feel free to change these
values_a = gap_latest[gap_latest.continent == continent_a].lifeExp.values
values_b = gap_latest[gap_latest.continent == continent_b].lifeExp.values

result_a = bootstrap_ci(values_a)
result_b = bootstrap_ci(values_b)

# TODO 3: plot both as error bars (mean +/- CI half-width) on one chart
means = [result_a[0], result_b[0]]
lower_err = [result_a[0] - result_a[1], result_b[0] - result_b[1]]
upper_err = [result_a[2] - result_a[0], result_b[2] - result_b[0]]

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.errorbar([continent_a, continent_b], means, yerr=[lower_err, upper_err],
            fmt="o", markersize=8, capsize=6, color="#3b6ea5")
ax.set_ylabel("Mean life expectancy (2007)")
ax.set_title("95% bootstrap CI of mean lifeExp")
plt.show()

print(f"{continent_a}: mean={result_a[0]:.2f}, 95% CI=({result_a[1]:.2f}, {result_a[2]:.2f})")
print(f"{continent_b}: mean={result_b[0]:.2f}, 95% CI=({result_b[1]:.2f}, {result_b[2]:.2f})")


**Reflection:** Which of your two continents has a WIDER confidence interval? Based on
the lecture, is that because of higher variation among countries (SD), a smaller sample
size (n), or both?

_Your answer:_

Africa has the wider confidence interval. Africa has both a larger number of countries AND much more
spread in life expectancy across those countries (some countries in the 40s, others in the 70s),
whereas Europe's countries are much more tightly clustered together in life expectancy. Per the
lecture, the CI width depends on the standard deviation of the underlying values (wider SD = wider
CI) and shrinks as n grows, so the two effects actually pull in opposite directions here — Africa has
more countries (larger n, which should narrow the CI) but far higher variation between countries
(larger SD, which widens it). Since Africa's CI still ends up wider despite the larger sample size,
that tells us the effect is being driven mainly by the higher variation (SD) among African countries,
not by sample size.


## Task 4 — Build the Morning Dashboard (Complaint #4)

**Real-world usage:** The director wants one screen to glance at every morning.

**Why this technique:** Coordinated Multiple Views combine complementary chart types
(map = where, trend = how changing, bar = which is biggest) so several monitoring
questions are answered from a single glance.

**TODO:** Build a 1x3 dashboard:
1. A map-style scatter of country centroids, colored by `lifeExp`.
2. A line chart of world-average `lifeExp` by year (use `gapminder.groupby("year")`).
3. A horizontal bar chart of total population by continent in 2007, sorted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# TODO 1: map-style scatter (axes[0])
sc = axes[0].scatter(gap_latest["Longitude"], gap_latest["Latitude"],
                      c=gap_latest["lifeExp"], cmap="viridis", s=25)
axes[0].set_title("Life expectancy by country (2007)")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
fig.colorbar(sc, ax=axes[0], shrink=0.8, label="lifeExp")

# TODO 2: world trend line (axes[1])
world_trend = gapminder.groupby("year")["lifeExp"].mean()
axes[1].plot(world_trend.index, world_trend.values, "o-", color="#3b6ea5")
axes[1].set_title("World average life expectancy over time")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Mean lifeExp")

# TODO 3: population by continent bar chart (axes[2])
pop_by_continent = gap_latest.groupby("continent")["pop"].sum().sort_values()
axes[2].barh(pop_by_continent.index, pop_by_continent.values, color="#3b6ea5")
axes[2].set_title("Total population by continent (2007)")
axes[2].set_xlabel("Population")

plt.tight_layout()
plt.show()


**Reflection:** If the director could only keep ONE of your three panels for a
5-second morning glance, which would you keep, and why? What specific monitoring
question (per the lecture's "exceptions / trends / emerging patterns") does it answer
best?

_Your answer:_

I'd keep the world-average life expectancy trend line. In a 5-second glance, the map panel takes a
moment to scan for outliers, and the population bar chart is more of a "which is biggest" reference
than something that changes day to day. The trend line, on the other hand, immediately answers the
"trends" monitoring question from the lecture — is life expectancy globally going up, flat, or (in a
future update) suddenly dropping — which is exactly the kind of high-level "is something changing"
signal a director wants before diving into the details of any one panel.


## Task 5 — The Decision (No Code)

**Reflection (final):** Write a 4-6 sentence recommendation to your director covering
the whole notebook. Your answer should:
- State which map framing (count vs. rate) you'd standardize on for all future reports,
  and why.
- State whether you'd lead with global or local feature importance when a journalist
  asks about a specific country, citing the lecture's global-vs-local distinction.
- Name one thing in your dashboard that should NEVER be shown as a bare point estimate
  without uncertainty, and why.

_Your final recommendation here:_

I'd standardize on rate-based choropleths (like GDP per capita) rather than raw counts for all future
reports, since count maps are dominated by population size and end up telling readers "this country
has a lot of people" instead of "this country is doing well" — which is exactly the Greenland-vs-India
trap the director already flagged. When a journalist asks about a specific country, I'd lead with
LOCAL feature importance (SHAP) rather than the model's global feature importance, because per the
lecture, global importance only tells you what matters on average across every row, while local
importance is the only thing that can actually answer a question about one specific country's
prediction. Finally, the continent-level life expectancy averages in our dashboard should never be
shown as a bare point estimate without a confidence interval, since it's computed from a small sample
of countries per continent and a single mean number would make a genuinely uncertain estimate look
like a hard fact, which is precisely the "look like estimates, not certainties" complaint the director
raised.
